# 02 · Controlled examples

The bundled toy examples pair a log with models whose language relation to the log is known by construction. They are the witness cases of the paper's controlled evaluation and show how LMP behaves on each precision axiom. Sections A2 to A5 follow the axioms of Tax et al. as numbered in the paper; the first section is the fitness-separation experiment, and A1 (determinism) is covered in [tutorial 07](07_precision_axioms_vs_pm4py.ipynb).

In [1]:
import os, warnings
os.environ["PM4PY_SHOW_PROGRESS_BAR"] = "False"
warnings.filterwarnings("ignore", category=DeprecationWarning)

import language_mass_precision as lmp
import pm4py
import pandas as pd
pd.set_option("display.precision", 6)

In [2]:
pd.Series(lmp.datasets.list_toys(), name="description").to_frame()

,description
precision_fitness,Precision vs fitness: 'smaller' misses a log v...
flower,Overgeneralization: a model constrained to the...
silent_routing,Language equivalence: a direct choice and a si...
log_extension,"Adding fitting behaviour: one model, three nes..."
decoy_stress,Language inclusion with structural decoys: 'wi...


## Precision is not fitness

`smaller` cannot replay one log variant, `exact` has the log's language, `larger` adds unseen behaviour. Only the last one loses precision: a model that under-fits is perfectly precise, it simply has low fitness.

In [3]:
toy = lmp.datasets.load_toy("precision_fitness")
print(toy.log.groupby("case:concept:name")["concept:name"].agg(" ".join).value_counts())

table = lmp.rank_models(toy.log, toy.models)[["model", "value", "accepted_variants", "rejected_variants"]]
table["token_fitness"] = [
    pm4py.fitness_token_based_replay(toy.log, *toy.models[m])["log_fitness"] for m in table["model"]
]
table

concept:name
a b c    3
a d c    3
Name: count, dtype: int64


,model,value,accepted_variants,rejected_variants,token_fitness
0,smaller,1.0,1,1,0.857143
1,exact,1.0,2,0,1.000000
2,larger,0.5,2,0,1.000000


## A2 · Language inclusion, even with structural decoys

`with_decoys` has exactly the log language but carries visible branches that deadlock. These decoys inflate replay-based measures yet add nothing to the language, so LMP ignores them. `larger` is clean but strictly larger, and scores lower.

In [4]:
toy = lmp.datasets.load_toy("decoy_stress")
table = lmp.rank_models(toy.log, toy.models)[["model", "value", "dfa_states"]]
table["token_precision"] = [pm4py.precision_token_based_replay(toy.log, *toy.models[m]) for m in table["model"]]
table

,model,value,dfa_states,token_precision
0,with_decoys,1.000000,16,0.3125
1,larger,0.842782,11,1.0000


## A3 · The flower model

A flower model accepts every trace over its alphabet. Its language mass is the whole of $\Sigma^*$, so a log with a single variant supports only a sliver of it.

In [5]:
toy = lmp.datasets.load_toy("flower")
lmp.rank_models(toy.log, toy.models)[["model", "status", "value", "certification"]]

,model,status,value,certification
0,constrained,exact,1.000000,full_language_finite_sum
1,flower,exact,0.001478,full_language_sparse_solve


Bounded playout, the classical approximation, enumerates only traces up to length *K*. It misses the infinite tail of the flower language and over-estimates precision; the exact computation needs no *K*.

In [6]:
exact = lmp.precision_language_mass(toy.log, *toy.models["flower"]).value
rows = []
for k in (3, 4, 5, 6, 7, 8):
    bounded = lmp.precision_language_mass(toy.log, *toy.models["flower"], method="playout", max_trace_length=k)
    rows.append({"K": k, "bounded": bounded.value, "exact": exact, "relative_error": bounded.value / exact - 1})
pd.DataFrame(rows)

,K,bounded,exact,relative_error
0,3,0.002956,0.001478,1.000000
1,4,0.002551,0.001478,0.725471
2,5,0.002287,0.001478,0.546918
3,6,0.002104,0.001478,0.423086
4,7,0.001971,0.001478,0.333333
5,8,0.001872,0.001478,0.266182


## A4 · Language equivalence

`direct` and `silent` are structurally different (one routes the choice through silent transitions) but have the same visible language. LMP is a function of the language only, so the two values are identical to the last digit.

In [7]:
toy = lmp.datasets.load_toy("silent_routing")
frame = lmp.rank_models(toy.log, toy.models, length_prior="power_law")
frame[["model", "status", "lower", "upper", "dfa_states"]]

,model,status,lower,upper,dfa_states
0,direct,exact,1.0,1.0,4
1,silent,exact,1.0,1.0,4


## A5 · Adding fitting behaviour

With the model and prior fixed, logs `L1 ⊂ L2 ⊂ L3` cover more and more of the model language. The numerator grows and LMP rises monotonically. The alphabet is declared so that the three scores share one prior.

In [8]:
toy = lmp.datasets.load_toy("log_extension")
estimator = lmp.LanguageMassPrecision(alphabet="abcde")
model = estimator.compile(*toy.models["choice"])
pd.DataFrame([
    {"log": name, "variants": len(estimator.fit(log).variants_), "LMP": estimator.score(model).value}
    for name, log in toy.logs.items()
])

,log,variants,LMP
0,L1,1,0.333333
1,L2,2,0.666667
2,L3,3,1.000000
